# Practica de Programacion y Algoritmia Avanzada – GIA

El objetivo de esta actividad es poner en practica los conocimientos de programacion dinamica, para tal caso, se os propone implementar el algoritmo determinista Cocke–Kasami–Younger.

El algoritmo CKY determinista es un algoritmo utilizado para analizar la estructura sintactica de cadenas de texto, y determinar si pueden ser derivadas apartir de las reglas de una gramatica.

In [ ]:
def cargar_gramatica(ruta_archivo):
    reglas = {}

    with open(ruta_archivo) as archivo:
        for linea in archivo:
            linea = linea.strip()
            if '->' in linea:
                lhs, rhs = linea.split('->')
                lhs = lhs.strip()
                alternativas = rhs.split('|')

                for alt in alternativas:
                    produccion = alt.strip().split()
                    if produccion:
                        reglas.setdefault(lhs, []).append(produccion)

    return reglas


In [ ]:
# gramatica= cargar_gramatica('c:/Users/veron/Documents/UNI/2º/Febrero/PAA/Practica/input.txt')

In [4]:
def es_cnf(gramatica):
    no_terminales = set(gramatica.keys())
    terminales = set()

    # Recorremos toda la gramática para ver qué símbolos podrían ser terminales
    for reglas in gramatica.values():
        for produccion in reglas:
            for simbolo in produccion:
                if simbolo not in no_terminales:
                    terminales.add(simbolo)

    # Ahora comprobamos si cada producción cumple las reglas de la CNF
    for lado_izq, reglas in gramatica.items():
        for produccion in reglas:
            if len(produccion) == 1:
                # Caso A -> a
                if produccion[0] not in terminales:
                    return False
            elif len(produccion) == 2:
                # Caso A -> B C
                if produccion[0] in terminales or produccion[1] in terminales:
                    return False
            else:
                # Producción con más de 2 símbolos ej A-> BCD
                return False

    return True


In [5]:
def cky(gramatica, palabra):
    n = len(palabra)
    tabla = [[set() for _ in range(n)] for _ in range(n)]

    # Crear un diccionario inverso: RHS -> LHS
    inversa = {}
    for lhs, producciones in gramatica.items():
        for rhs in producciones:
            clave = tuple(rhs)
            inversa.setdefault(clave, []).append(lhs)

    # Rellenar la diagonal (longitud 1): A -> a
    for i, simbolo in enumerate(palabra):
        if (simbolo,) in inversa:
            tabla[i][i].update(inversa[(simbolo,)])

    # Rellenar el resto de la tabla
    for longitud in range(2, n + 1):  # tamaño del fragmento
        for i in range(n - longitud + 1):
            j = i + longitud - 1
            for k in range(i, j):
                izquierda = tabla[i][k]
                derecha = tabla[k + 1][j]
                for B in izquierda:
                    for C in derecha:
                        if (B, C) in inversa:
                            tabla[i][j].update(inversa[(B, C)])

    # Verificamos si el símbolo inicial S genera toda la cadena
    return 'S' in tabla[0][n - 1]


In [ ]:
# Ejemplo de uso
# Usamos las dos gramáticas proporcionadas en el enunciado

Gramar1 = {
    'S': [['a'], ['X', 'A'], ['A', 'X'], ['b']],
    'A': [['R', 'B']],
    'B': [['A', 'X'], ['b'], ['a']],
    'X': [['a']],
    'R': [['X', 'B']]
}

Gramar2 = {
    'S': [['A','B'], ['C','D'], ['C','B'], ['S','S']],
    'A': [['B','C'], ['a']],
    'B': [['S','C'], ['b']],
    'C': [['D','D'], ['b']],
    'D': [['B','A']]
}

print("Grammar in CNF:", es_cnf(Gramar1)) # True
print("Grammar in CNF:", es_cnf(Gramar2)) # True

print("Can generate 'abab':", cky(Gramar1, ['a', 'b', 'a', 'b'])) # False
print("Can generate 'abba':", cky(Gramar1, ['a', 'b', 'b', 'a'])) # True



Grammar in CNF: True
Grammar in CNF: True
Can generate 'abab': False
Can generate 'abba': True


In [34]:
# EXTENSION 1
# Transformacion de cualquier gramatica CFG a CNF.

# Funciones auxiliares para la transformación a CNF

def eliminar_unitarias(gramatica):
    """Elimina las producciones unitarias de la gramática."""
    unitarias = set()
    
    # Encontramos las producciones unitarias
    for A, reglas in gramatica.items():
        for regla in reglas:
            if len(regla) == 1 and regla[0].isupper():
                unitarias.add((A, regla[0]))
    
    # Sustituir las producciones unitarias
    for A, B in unitarias:
        for regla_B in gramatica[B]:
            if regla_B not in gramatica[A]:
                gramatica[A].append(regla_B)
    
    # Eliminar las producciones unitarias
    for A, B in unitarias:
        gramatica[A] = [regla for regla in gramatica[A] if regla != [B]]
    
    return gramatica


def eliminar_vacias(gramatica):
    """Elimina las producciones vacías (A -> ε)."""
    nullable = set()

    # Encontramos las variables que pueden generar ε
    for A, reglas in gramatica.items():
        for regla in reglas:
            if regla == []:
                nullable.add(A)

    # Eliminar las producciones vacías y generar nuevas producciones
    cambios = True
    while cambios:
        cambios = False
        for A in list(gramatica):
            # Eliminar las producciones vacías
            nuevas_reglas = [regla for regla in gramatica[A] if regla != []]
            for regla in gramatica[A]:
                for i in range(len(regla)):
                    if regla[i] in nullable:
                        nueva_regla = regla[:i] + regla[i+1:]
                        if nueva_regla not in nuevas_reglas:
                            nuevas_reglas.append(nueva_regla)
                            
            gramatica[A] = nuevas_reglas
            for i in gramatica[A]:
                if i == []:
                    cambios = True
                    break
    return gramatica


def eliminar_inutiles(gramatica):
    """Elimina las producciones inútiles (no alcanzables o no generativas)."""
    generativas = set()
    alcanzables = set([list(gramatica.keys())[0]])  # Empezamos con el símbolo inicial

    # Encontramos las variables generativas
    while True:
        nueva_generativa = set(generativas)
        for A, reglas in gramatica.items():
            if A not in generativas:
                for regla in reglas:
                    if all(simbolo.islower() or simbolo in generativas for simbolo in regla):
                        nueva_generativa.add(A)
                        break
        if nueva_generativa == generativas:
            break
        generativas = nueva_generativa

    # Encontramos las variables alcanzables
    while True:
        nueva_alcanzable = set(alcanzables)
        for A in alcanzables:
            for regla in gramatica.get(A, []):
                for simbolo in regla:
                    if simbolo.isupper() and simbolo not in alcanzables:
                        nueva_alcanzable.add(simbolo)
        if nueva_alcanzable == alcanzables:
            break
        alcanzables = nueva_alcanzable

    # Eliminar producciones inútiles
    gramatica = {A: [regla for regla in reglas if all(simbolo in generativas and (simbolo in alcanzables or simbolo.islower()) for simbolo in regla)]
                 for A, reglas in gramatica.items() if A in generativas}

    return gramatica


def reemplazar_terminales(gramatica):
    """Reemplaza los terminales por nuevas variables (A -> a -> X)."""
    terminales_a_variables = {}
    nuevo_contador = 0

    for A, reglas in gramatica.items():
        for regla in reglas:
            for i in range(len(regla)):
                if isinstance(regla[i], str) and regla[i].islower():  # Si el símbolo es un terminal
                    if regla[i] not in terminales_a_variables:
                        nuevo_simbolo = f"X{nuevo_contador}"
                        terminales_a_variables[regla[i]] = nuevo_simbolo
                        gramatica[nuevo_simbolo] = [[regla[i]]]  # Crear una nueva producción para el terminal
                        nuevo_contador += 1
                    regla[i] = terminales_a_variables[regla[i]]  # Reemplazar el terminal por la nueva variable

    return gramatica


def reducir_terminales(gramatica):
    """Convierte producciones con más de dos símbolos en producciones válidas en CNF."""
    nuevas_producciones = {}
    nuevo_contador = 0

    # Recorremos todas las producciones
    for A, reglas in gramatica.items():
        nuevas_producciones[A] = []
        for regla in reglas:
            if len(regla) > 2:
                # Crear nuevas variables para producciones con más de dos símbolos
                nuevo_simbolo = f"X{nuevo_contador}"
                nuevo_contador += 1
                nuevas_producciones[nuevo_simbolo] = [regla[1:]]  # La parte de la regla a partir del segundo símbolo
                nuevas_producciones[A].append([regla[0], nuevo_simbolo])  # Se reemplaza por la nueva variable
            else:
                nuevas_producciones[A].append(regla)
    
    return nuevas_producciones


In [ ]:
# Funcion principal para transformar una gramática CFG a CNF

def cfg_a_cnf(gramatica):
    """Convierte una gramática CFG a CNF aplicando los pasos necesarios."""
    # Paso 1: Eliminar producciones vacías (repetir hasta que no haya cambios)
    gramatica = eliminar_vacias(gramatica)
    
    # Paso 2: Eliminar producciones unitarias
    gramatica = eliminar_unitarias(gramatica)
    
    # Paso 3: Eliminar producciones inútiles
    gramatica = eliminar_inutiles(gramatica)
    
    # Paso 4: Reemplazar terminales por variables
    gramatica = reemplazar_terminales(gramatica)
    
    # Paso 5: Reducir producciones con más de dos símbolos
    gramatica = reducir_terminales(gramatica)
    
    return gramatica


In [36]:
# Ejemplo de uso de la transformación a CNF

Grammar_CFG = {
    'S': [['A', 'S', 'B']],
    'A': [['a', 'A', 'S'], ['a'], []],  
    'B': [['S', 'b', 'S'], ['A'], ['b', 'b']]
}

# Resultado esperado
# Grammar_CFG_CNF = {
#     'S': [['A', 'S'], ['P', 'B'], ['S', 'B']],
#     'A': [['R', 'S'], ['X', 'S'], ['a']],
#     'B': [['T', 'S'], ['Y', 'Y'], ['R', 'S'], ['X', 'S'], ['a']],
#     'X': [['a']],
#     'Y': [['b']],
#     'P': [['A', 'S']],
#     'R': [['X', 'A']],
#     'T': [['S', 'Y']]
# }


print("Grammar in CNF:", es_cnf(Grammar_CFG))

print("Original Grammar:", Grammar_CFG, end="\n\n")
print("Paso 1:", eliminar_vacias(Grammar_CFG), end="\n\n") # eliminar hasta que no queden
print("Paso 2:", eliminar_unitarias(Grammar_CFG), end="\n\n")
print("Paso 3:", eliminar_inutiles(Grammar_CFG), end="\n\n")
# print("Paso 4:", reemplazar_terminales(Grammar_CFG), end="\n\n")
# print("Paso 5:", reducir_terminales(Grammar_CFG), end="\n\n")



Grammar in CNF: False
Original Grammar: {'S': [['A', 'S', 'B']], 'A': [['a', 'A', 'S'], ['a'], []], 'B': [['S', 'b', 'S'], ['A'], ['b', 'b']]}

Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "C:\Users\Claudia\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\IPython\core\interactiveshell.py", line 3672, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "C:\Users\Claudia\AppData\Local\Temp\ipykernel_54452\3973793064.py", line 25, in <module>
    print("Paso 1:", eliminar_vacias(Grammar_CFG), end="\n\n") # eliminar hasta que no queden
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Claudia\AppData\Local\Temp\ipykernel_54452\458767320.py", line None, in eliminar_vacias
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\Claudia\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\IPython\core\interactiveshell.py", line 2178, in showtraceback
    stb = self.Interact

In [ ]:
Grammar_CNF = cfg_a_cnf(Grammar_CFG)
print("Transformed Grammar to CNF:", Grammar_CNF)

In [ ]:
# EXTENSION 2
# ampliar el algoritmo CKY a probabilıstico.

